# Model Inference and Robot Evaluation

## ▶️ Action needed — this is the hands-on notebook

Notebooks 2 and 3 were background reading; the scene, the task, and the environment are already
installed. Here you'll run two commands in two terminals and watch the robot attempt the
pick-and-place task in Isaac Sim, driven by a fine-tuned GR00T N1.5 policy.

## Overview

The pieces you read about now come together:

- A **policy server** holds the fine-tuned GR00T N1.5 model and answers "given these camera images
  and joint positions, what should the arm do next?"
- The **simulation** runs `LeIsaac-SO101-PickOrange-v0`, sends observations to that server, and
  applies the actions it returns.

They are two separate processes that talk over a socket on port `5555`, so you need **two
terminals**, started in this order.

## Step 1: Start the inference server

1. Open a terminal.
2. Activate the GR00T environment and go to the GR00T directory:

   ```bash
   conda activate gr00t
   cd ~/workspace/gr00t
   ```

3. Launch the server:

   ```bash
   python scripts/inference_service.py \
       --model-path flrs/so101_orange_pick_gr00tn1.5_model \
       --server \
       --embodiment_tag new_embodiment \
       --data-config so100_dualcam
   ```

What each part does:

- `--model-path` — a public Hugging Face repo holding a GR00T N1.5 model fine-tuned on
  pick-orange demonstrations. On first run it downloads (a few GB) and caches under
  `~/.cache/huggingface`; it should already be warm on this machine, so expect it to load from
  cache. You can also pass a local directory here instead.
- `--server` — run as a server and listen on port `5555` instead of doing a one-shot inference.
- `--embodiment_tag new_embodiment` — the SO-101 arm was **not** part of GR00T N1.5's
  pre-training, so it is treated as a new embodiment (this has to match how the model was
  fine-tuned).
- `--data-config so100_dualcam` — the two-camera layout from
  [notebook 3](03_sim_task.ipynb): `wrist` + `front`.

**Wait until the server reports it is ready and listening before continuing.** It has to load a
3-billion-parameter model onto the GPU, which takes a while. **Leave this terminal running** — if
you close it, the simulation will have nothing to talk to.

## Step 2: Run policy inference in the simulation

1. Open a **new** terminal (leave the server running in the first one).
2. Activate the LeIsaac environment and go to the LeIsaac directory:

   ```bash
   conda activate leisaac
   cd ~/workspace/leisaac
   ```

3. Start the evaluation:

   ```bash
   python scripts/evaluation/policy_inference.py \
       --task=LeIsaac-SO101-PickOrange-v0 \
       --eval_rounds=10 \
       --policy_type=gr00tn1.5 \
       --policy_host=localhost \
       --policy_port=5555 \
       --policy_timeout_ms=5000 \
       --policy_action_horizon=16 \
       --policy_language_instruction="Pick up the orange and place it on the plate" \
       --device=cuda \
       --enable_cameras
   ```

### Parameter explanation

- `--task` — the Gymnasium id registered in `tasks/pick_orange/__init__.py`.
- `--eval_rounds` — how many attempts to run (10). Remember each round re-randomises the orange,
  plate, and camera positions, so the rounds are genuinely different.
- `--policy_type` — which client to use to talk to the server; `gr00tn1.5` matches the server you
  started in step 1.
- `--policy_host` / `--policy_port` — where that server is listening.
- `--policy_timeout_ms` — how long to wait for a reply before giving up.
- `--policy_action_horizon` — how many future actions to take from each prediction (16). The model
  returns a chunk of actions per inference call, rather than one step at a time.
- `--policy_language_instruction` — the natural-language goal. GR00T is a **vision-language**-action
  model: this text is a real input, not a label. Try editing it later and see whether behaviour
  changes.
- `--device` — run the simulation on the GPU.
- `--enable_cameras` — render the `wrist` and `front` cameras. Without this the policy would be
  blind and the run would fail.

Isaac Sim will open, load the kitchen, and the arm will start moving on its own.

## Step 3: Watch it, from different angles

The default view is a free perspective camera. To see what the *policy* sees, click the **camera
icon** in the viewport toolbar and switch from `Perspective` to `camera/wrist` or `camera/front`.

The wrist view is the interesting one — it's a close-up that swings with the gripper, and it makes
clear how little context the policy has to work with when it decides to close the jaws.

## What to look for

- Does it reach for an orange, or hesitate?
- Does the grasp hold, or does the orange squirt out of the gripper?
- Does it manage all **three** oranges, or lose track after the first?
- Over 10 rounds, how often does it succeed? Remember the layout shifts each round — which shifts
  break it?

Keep your answers in mind for [notebook 5](05_challenges.ipynb).

## Troubleshooting

**Red `PhysicsUSD: CreateJoint - cannot create a joint between static bodies` errors on startup.**
Expected and harmless — see [notebook 2](02_sim_scene.ipynb). There will be dozens, one per wall
and counter. The scene still loads.

**Warnings at the very end of the output**, such as:

```text
[Warning] [omni.fabric.plugin] gFabricState->gUsdStageToSimStageWithHistoryMap had 1 outstanding
  SimStageWithHistory(s) at shutdown
[Warning] [carb] Recursive unloadAllPlugins() detected!
```

These are **shutdown noise**, printed while Isaac Sim tears itself down. They are not the reason
anything failed. If the command exited unexpectedly, the real cause is a Python `Traceback`
**further up** in the output — scroll back to find it. (Piping to a file helps:
`... --enable_cameras 2>&1 | tee ~/inference.log`, then `grep -A25 Traceback ~/inference.log`.)

**The simulation starts but the arm never moves.** The server in step 1 probably isn't ready or
isn't reachable. Check that terminal for errors, and confirm it's listening:

```bash
ss -ltn | grep 5555
```

**Out of GPU memory.** The 3B model and Isaac Sim share one GPU. Close any other Isaac Sim window
first, and check with `nvidia-smi`.